In [1]:
import pandas as pd
import numpy as np


# Notebook pour le Remplissage des NA

In [2]:
df_ventes = pd.read_csv('data/ech_annonces_ventes_68.csv', sep=';', index_col='idannonce')

## Etape 1 - Début: Nettoyage des colonnes

#### On enlève du df_ventes les colonnes avec trop (plus de 80%) de variables manquantes

In [3]:
valeurs_manquantes = df_ventes.isna().sum()/df_ventes.shape[0]
valeurs_manquantes_list = list(valeurs_manquantes[valeurs_manquantes > 0.8].index)
valeurs_manquantes_list

['prix_maison',
 'prix_terrain',
 'parking',
 'nb_terraces',
 'videophone',
 'porte_digicode',
 'surface_balcon']

In [4]:
df_ventes_filtered = df_ventes.drop(valeurs_manquantes_list, axis = 1)
df_ventes_filtered.shape

(27737, 51)

In [5]:
column_n6 = [ column for column in df_ventes_filtered.columns if "n6" in column]
column_n6

['loyer_m2_median_n6', 'nb_log_n6', 'taux_rendement_n6']

#### Précédemment nous avions dit que nous gardions plutôt les varaibles suffixées _n7 basées sur un plus grand nombre de logements et abandonnons les variables n6

In [6]:
df_ventes_filtered = df_ventes_filtered.drop(column_n6, axis = 1)

In [7]:
df_ventes_filtered.shape

(27737, 48)

#### On va beaucoup s'appuyer sur la colonne typedebien, on harmonise ses valeurs

In [8]:
df_ventes_filtered['typedebien'].value_counts()

typedebien
m     13288
a     13158
mn      650
an      640
l         1
Name: count, dtype: int64

##### On enlève le seul l (lot?) et on met tous les appartements en a et maison en m

In [9]:
df_ventes_filtered = df_ventes_filtered[df_ventes_filtered['typedebien'] != 'l']
# Verification
df_ventes_filtered.shape

(27736, 48)

In [10]:
df_ventes_filtered['typedebien'] = df_ventes_filtered['typedebien'].replace({'an': 'a', 'mn': 'm'})
# Verification
df_ventes_filtered['typedebien'].value_counts()

typedebien
m    13938
a    13798
Name: count, dtype: int64

In [11]:
df_ventes_filtered.shape

(27736, 48)

In [12]:
df_ventes_filtered.info()

<class 'pandas.core.frame.DataFrame'>
Index: 27736 entries, hektor-robischung-2862 to 139483195
Data columns (total 48 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   type_annonceur           27736 non-null  object 
 1   typedebien               27736 non-null  object 
 2   typedetransaction        27736 non-null  object 
 3   etage                    27736 non-null  int64  
 4   surface                  27736 non-null  int64  
 5   surface_terrain          11444 non-null  float64
 6   nb_pieces                27736 non-null  int64  
 7   prix_bien                27736 non-null  int64  
 8   mensualiteFinance        27736 non-null  int64  
 9   balcon                   27736 non-null  int64  
 10  eau                      27736 non-null  int64  
 11  bain                     27736 non-null  int64  
 12  dpeL                     27736 non-null  object 
 13  dpeC                     17150 non-null  float64
 14  ma

###### On identifie les variables quanti et celles quali avec des valeurs manquantes

In [13]:
valeurs_manq_resid_quanti = [col for  col in df_ventes_filtered.select_dtypes(exclude='object').columns if df_ventes_filtered[col].isna().sum() > 0]
valeurs_manq_resid_quanti

['surface_terrain',
 'dpeC',
 'nb_etages',
 'places_parking',
 'annee_construction',
 'nb_toilettes',
 'nb_logements_copro',
 'charges_copro',
 'duree_int',
 'loyer_m2_median_n7',
 'nb_log_n7',
 'taux_rendement_n7']

In [14]:
valeurs_manq_resid_quali = [col for  col in df_ventes_filtered.select_dtypes(include='object').columns if df_ventes_filtered[col].isna().sum() > 0]
valeurs_manq_resid_quali

['cave',
 'ges_class',
 'ascenseur',
 'chauffage_energie',
 'chauffage_systeme',
 'chauffage_mode',
 'logement_neuf']

In [15]:
print(f"Nombre de colonnes avec données manquantes quanti: {len(valeurs_manq_resid_quanti)}\nNombre de colonnes avec données manquantes quanti: {len(valeurs_manq_resid_quali)}")

Nombre de colonnes avec données manquantes quanti: 12
Nombre de colonnes avec données manquantes quanti: 7


### verification de certaines colonnes qui sont en float mais devraient être plutôt en int

In [16]:
col_float = df_ventes_filtered[valeurs_manq_resid_quanti].select_dtypes(include='float64').columns

In [17]:
col_int = []
for col in col_float:
    array_col = np.array(df_ventes_filtered[df_ventes_filtered[col].notna()][col]) 
    array_col_round = np.round(array_col)
    array_real_float = array_col[array_col != array_col_round]
    print(f"Col: {col}, Nombre d'éléments non Na: {len(array_col)}, nombre d'éléments non entiers: {len(array_real_float)}")
    if len(array_real_float) == 0:
        col_int.append(col)

Col: surface_terrain, Nombre d'éléments non Na: 11444, nombre d'éléments non entiers: 155
Col: dpeC, Nombre d'éléments non Na: 17150, nombre d'éléments non entiers: 228
Col: nb_etages, Nombre d'éléments non Na: 14859, nombre d'éléments non entiers: 0
Col: places_parking, Nombre d'éléments non Na: 10169, nombre d'éléments non entiers: 0
Col: annee_construction, Nombre d'éléments non Na: 9458, nombre d'éléments non entiers: 0
Col: nb_toilettes, Nombre d'éléments non Na: 15436, nombre d'éléments non entiers: 0
Col: nb_logements_copro, Nombre d'éléments non Na: 7565, nombre d'éléments non entiers: 0
Col: charges_copro, Nombre d'éléments non Na: 6677, nombre d'éléments non entiers: 722
Col: duree_int, Nombre d'éléments non Na: 26195, nombre d'éléments non entiers: 0
Col: loyer_m2_median_n7, Nombre d'éléments non Na: 19650, nombre d'éléments non entiers: 17842
Col: nb_log_n7, Nombre d'éléments non Na: 19650, nombre d'éléments non entiers: 0
Col: taux_rendement_n7, Nombre d'éléments non Na: 1

In [18]:
col_int

['nb_etages',
 'places_parking',
 'annee_construction',
 'nb_toilettes',
 'nb_logements_copro',
 'duree_int',
 'nb_log_n7']

In [19]:
for col in col_int:
    df_ventes_filtered[col] = df_ventes_filtered[col].astype('Int64')

In [21]:
df_ventes_filtered[valeurs_manq_resid_quanti].info()

<class 'pandas.core.frame.DataFrame'>
Index: 27736 entries, hektor-robischung-2862 to 139483195
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   surface_terrain     11444 non-null  float64
 1   dpeC                17150 non-null  float64
 2   nb_etages           14859 non-null  Int64  
 3   places_parking      10169 non-null  Int64  
 4   annee_construction  9458 non-null   Int64  
 5   nb_toilettes        15436 non-null  Int64  
 6   nb_logements_copro  7565 non-null   Int64  
 7   charges_copro       6677 non-null   float64
 8   duree_int           26195 non-null  Int64  
 9   loyer_m2_median_n7  19650 non-null  float64
 10  nb_log_n7           19650 non-null  Int64  
 11  taux_rendement_n7   19650 non-null  float64
dtypes: Int64(7), float64(5)
memory usage: 3.9+ MB


#### Nous allons tansformer certaines colonnes object en fait booleenne en int

In [22]:
df_ventes_filtered[valeurs_manq_resid_quali].describe()

,cave,ges_class,ascenseur,chauffage_energie,chauffage_systeme,chauffage_mode,logement_neuf
count,13247,19716,8750,12571,7627,13610,26997
unique,2,9,2,10,13,6,2
top,True,D,True,Gaz,Radiateur,Individuel,n
freq,8626,3943,4994,7787,6254,11186,23391


In [23]:
unique = df_ventes_filtered[valeurs_manq_resid_quali].describe().loc['unique']
col_bool = list(unique[unique ==2].index)
col_bool

['cave', 'ascenseur', 'logement_neuf']

In [24]:
for col in col_bool:
    print(f"{df_ventes_filtered[col].value_counts()}")

cave
True     8626
False    4621
Name: count, dtype: int64
ascenseur
True     4994
False    3756
Name: count, dtype: int64
logement_neuf
n    23391
o     3606
Name: count, dtype: int64


In [25]:
df_ventes_filtered["cave"] = df_ventes_filtered["cave"] .astype('Int64')

In [26]:
df_ventes_filtered["ascenseur"] = df_ventes_filtered["ascenseur"] .astype('Int64')

In [27]:
df_ventes_filtered["logement_neuf"] = df_ventes_filtered["logement_neuf"].replace({'n': False, 'o': True}).astype('Int64')

In [28]:
for col in col_bool:
    print(f"{df_ventes_filtered[col].value_counts()}")

cave
1    8626
0    4621
Name: count, dtype: Int64
ascenseur
1    4994
0    3756
Name: count, dtype: Int64
logement_neuf
0    23391
1     3606
Name: count, dtype: Int64


In [29]:
df_ventes_filtered[col_bool].dtypes

cave             Int64
ascenseur        Int64
logement_neuf    Int64
dtype: object

## Etape 1 -  Cellule récapitulative regroupant toutes les étapes:  Générer le df avec colonnes nettoyées

In [1]:
import pandas as pd
import numpy as np

df_ventes = pd.read_csv('data/ech_annonces_ventes_68.csv', sep=';', index_col='idannonce')
valeurs_manquantes = df_ventes.isna().sum()/df_ventes.shape[0]
valeurs_manquantes_list = list(valeurs_manquantes[valeurs_manquantes > 0.8].index)
df_ventes_filtered = df_ventes.drop(valeurs_manquantes_list, axis = 1)
column_n6 = [ column for column in df_ventes_filtered.columns if "n6" in column]
df_ventes_filtered = df_ventes_filtered.drop(column_n6, axis = 1)
df_ventes_filtered = df_ventes_filtered[df_ventes_filtered['typedebien'] != 'l']
df_ventes_filtered['typedebien'] = df_ventes_filtered['typedebien'].replace({'an': 'a', 'mn': 'm'})
valeurs_manq_resid_quanti = [col for  col in df_ventes_filtered.select_dtypes(exclude='object').columns if df_ventes_filtered[col].isna().sum() > 0]

col_float = df_ventes_filtered[valeurs_manq_resid_quanti].select_dtypes(include='float64').columns
for col in col_float:
    array_col = np.array(df_ventes_filtered[df_ventes_filtered[col].notna()][col]) 
    array_col_round = np.round(array_col)
    array_real_float = array_col[array_col != array_col_round]
    if len(array_real_float) == 0:
        df_ventes_filtered[col] = df_ventes_filtered[col].astype('Int64')
df_ventes_filtered["cave"] = df_ventes_filtered["cave"].astype('Int64')
df_ventes_filtered["ascenseur"] = df_ventes_filtered["ascenseur"].astype('Int64')
df_ventes_filtered["logement_neuf"] = df_ventes_filtered["logement_neuf"].replace({'n': False, 'o': True}).astype('Int64')

In [2]:
df_ventes_filtered.shape

(27736, 48)

### Sauvegarde du df de l'étape 1

In [3]:
df_ventes_filtered.to_csv("df_col_cleaned.csv")

## Etape 1-bis: Séparation du df en train et test

#### Exemple de séparation train /test car process suivants se font avec un train et un test (pour apprendre et transformer sur le train et transformer uniquement sur le test)

In [4]:
X_train = df_ventes_filtered
X_test = pd.DataFrame(columns = X_train.columns)

## Etape 2 - Début: remplissage des colonnes quali

In [4]:
# Echantillon temoin pour vérifier les transformations des variables quali:
index_temoin = df_ventes_filtered[df_ventes_filtered["cave"].isna()]["cave"].index[0]
df_ventes_filtered.loc[index_temoin]

type_annonceur                     pr
typedebien                          m
typedetransaction                   v
etage                               0
surface                            85
surface_terrain                4950.0
nb_pieces                           4
prix_bien                      189000
mensualiteFinance                   0
balcon                              0
eau                                 0
bain                                0
dpeL                                0
dpeC                              NaN
mapCoordonneesLatitude       48.17073
mapCoordonneesLongitude       7.10498
annonce_exclusive                   0
nb_etages                        <NA>
places_parking                   <NA>
cave                             <NA>
exposition                        Sud
ges_class                         NaN
annee_construction               <NA>
nb_toilettes                     <NA>
ascenseur                        <NA>
nb_logements_copro               <NA>
charges_copr

##### Vérifications des variables quali et quanti

In [5]:
valeurs_manq_resid_quanti = [col for  col in df_ventes_filtered.select_dtypes(exclude='object').columns if df_ventes_filtered[col].isna().sum() > 0]
valeurs_manq_resid_quali = [col for  col in df_ventes_filtered.select_dtypes(include='object').columns if df_ventes_filtered[col].isna().sum() > 0]

In [6]:
print(f"Nombre de colonnes avec données manquantes quanti: {len(valeurs_manq_resid_quanti)}: {list(valeurs_manq_resid_quanti)}")
print(f"\nNombre de colonnes avec données manquantes quanti: {len(valeurs_manq_resid_quali)}: {list(valeurs_manq_resid_quali)}")

Nombre de colonnes avec données manquantes quanti: 15: ['surface_terrain', 'dpeC', 'nb_etages', 'places_parking', 'cave', 'annee_construction', 'nb_toilettes', 'ascenseur', 'nb_logements_copro', 'charges_copro', 'logement_neuf', 'duree_int', 'loyer_m2_median_n7', 'nb_log_n7', 'taux_rendement_n7']

Nombre de colonnes avec données manquantes quanti: 4: ['ges_class', 'chauffage_energie', 'chauffage_systeme', 'chauffage_mode']


##### On a moins de colonnes quali qu'au départ en ayant transformé les colonnes booléennes

##### On décide de faire une imputation via les plus proches voisins mais en se focalisant sur les biens de mêmes types étant proche au niveau localication et nombre de pièces


## Etape 2: Cellule récapitulative regroupant toutes les étapes:  Remplissage des colonnes quanti du X_train et X_test

In [2]:
from sklearn.impute import KNNImputer

## Si on veut faire directement l'étape 2, partir du fichier sauvé à l'étape 1 en enlevant les commentaires ci-dessous 
# import pandas as pd
# df_ventes_filtered = pd.read_csv("df_col_cleaned.csv", index_col = "idannonce")

# Remplacer ci-dessous la séparation pa un vrai split test/train. ici on prend le jeu de données entier comme exemple
X_train = df_ventes_filtered
X_test = pd.DataFrame(columns = X_train.columns)


neighbours_columns = ["nb_pieces", "mapCoordonneesLatitude", "mapCoordonneesLongitude"]
type_column = 'typedebien'

def df_imputed(input_df: pd.DataFrame, test_df: pd.DataFrame, col_processed, default):
    imputer = KNNImputer()
    cols_backup = input_df.columns
    index_backup = input_df.index
    test_index = test_df.index
    if input_df.shape[0] > 0:
        imputed_data = imputer.fit_transform(input_df)
        if imputed_data.shape[1] == len(cols_backup):
            imputed_df = pd.DataFrame(data=imputed_data, columns=cols_backup, index=index_backup)
            if test_df.shape[0] > 0:
                test_data = imputer.transform(test_df)
                imputed_test_df = pd.DataFrame(data=test_data, columns=cols_backup, index=test_index)
            else:
                imputed_test_df = test_df
        else: 
            # Case when the input can not have been done
            cols_wo_processed = cols_backup.drop(col_processed)
            imputed_df = pd.DataFrame(data=imputed_data, columns=cols_wo_processed, index=index_backup)
            imputed_df[col_processed] = default
            imputed_test_df = pd.DataFrame(data=test_df[cols_wo_processed], columns=cols_wo_processed, index=test_index)
            imputed_test_df[col_processed] = default
            
    else:
        imputed_data = input_df
        imputed_test_df = test_df
        
    return imputed_df, imputed_test_df


valeurs_manq_resid_quanti = [col for  col in df_ventes_filtered.select_dtypes(exclude='object').columns if df_ventes_filtered[col].isna().sum() > 0]

for col in valeurs_manq_resid_quanti:
    print(f"Processing column {col}")
    # On se focalise sur la colonne et nos colonnes voisines
    df_extract = X_train[[type_column] + [col] + neighbours_columns]
    df_test_extract = X_test[[type_column] + [col] + neighbours_columns]
    default_value = X_train[col].mean()
    # On distingue par type de bien
    df_extract_app = df_extract[df_extract[type_column] =='a'][[col] + neighbours_columns]
    df_extract_mais = df_extract[df_extract[type_column] =='m'][[col] + neighbours_columns]
    df_test_extract_app = df_test_extract[df_test_extract[type_column] =='a'][[col] + neighbours_columns]
    df_test_extract_mais = df_test_extract[df_test_extract[type_column] =='m'][[col] + neighbours_columns]
    
    df_filled_app, df_test_filled_app = df_imputed(df_extract_app, df_test_extract_app, col, default_value)
    df_filled_mais, df_test_filled_mais = df_imputed(df_extract_mais, df_test_extract_mais, col, default_value)
    df_filled = pd.concat([df_filled_app, df_filled_mais])
    df_test_filled = pd.concat([df_test_filled_app, df_test_filled_mais])

    if X_train[col].dtypes in ['int64', 'Int64']:
        X_train[col] = df_filled[col].apply(lambda x: round(x)).astype('int64')
        X_test[col] = df_test_filled[col].apply(lambda x: round(x)).astype('int64')
    else:
        X_train[col] = df_filled[col]
        X_test[col] = df_test_filled[col]

Processing column surface_terrain
Processing column dpeC
Processing column nb_etages
Processing column places_parking
Processing column cave
Processing column annee_construction
Processing column nb_toilettes
Processing column ascenseur
Processing column nb_logements_copro
Processing column charges_copro
Processing column logement_neuf
Processing column duree_int
Processing column loyer_m2_median_n7
Processing column nb_log_n7
Processing column taux_rendement_n7


##### Verification des modifications sur l echantillon temoin

In [5]:
df_ventes_filtered.loc[index_temoin]

NameError: name 'index_temoin' is not defined

In [4]:
df_ventes_filtered[valeurs_manq_resid_quanti].info()

<class 'pandas.core.frame.DataFrame'>
Index: 27736 entries, hektor-robischung-2862 to 139483195
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   surface_terrain     27736 non-null  float64
 1   dpeC                27736 non-null  float64
 2   nb_etages           27736 non-null  int64  
 3   places_parking      27736 non-null  int64  
 4   cave                27736 non-null  int64  
 5   annee_construction  27736 non-null  int64  
 6   nb_toilettes        27736 non-null  int64  
 7   ascenseur           27736 non-null  int64  
 8   nb_logements_copro  27736 non-null  int64  
 9   charges_copro       27736 non-null  float64
 10  logement_neuf       27736 non-null  int64  
 11  duree_int           27736 non-null  int64  
 12  loyer_m2_median_n7  27736 non-null  float64
 13  nb_log_n7           27736 non-null  int64  
 14  taux_rendement_n7   27736 non-null  float64
dtypes: float64(5), int64(10)
memory u

#### Il n'y a plus de données manquantes quanti

##### On sauvegarde en csv ce fichier étape avec les variables quanti remplies

In [3]:
df_ventes_filtered.to_csv("df_filled_quanti.csv")

## Etape 3 - Début: remplissage des colonnes quanti

##### On peut récuperer le df deja travaillé pour les variables quanti

In [19]:
import pandas as pd
df_ventes_filtered = pd.read_csv("df_filled_quanti.csv", index_col = "idannonce")

In [5]:
# Pour vérifier le type des colonnes récupérées via le csv
df_ventes_filtered.info()

<class 'pandas.core.frame.DataFrame'>
Index: 27736 entries, hektor-robischung-2862 to 139483195
Data columns (total 48 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   type_annonceur           27736 non-null  object 
 1   typedebien               27736 non-null  object 
 2   typedetransaction        27736 non-null  object 
 3   etage                    27736 non-null  int64  
 4   surface                  27736 non-null  int64  
 5   surface_terrain          27736 non-null  float64
 6   nb_pieces                27736 non-null  int64  
 7   prix_bien                27736 non-null  int64  
 8   mensualiteFinance        27736 non-null  int64  
 9   balcon                   27736 non-null  int64  
 10  eau                      27736 non-null  int64  
 11  bain                     27736 non-null  int64  
 12  dpeL                     27736 non-null  object 
 13  dpeC                     27736 non-null  float64
 14  ma

##### On inspecte les colonnes quali avec valeurs manquantes

In [6]:
valeurs_manq_resid_quali = [col for  col in df_ventes_filtered.select_dtypes(include='object').columns if df_ventes_filtered[col].isna().sum() > 0]

In [7]:
df_ventes_filtered[valeurs_manq_resid_quali].info()

<class 'pandas.core.frame.DataFrame'>
Index: 27736 entries, hektor-robischung-2862 to 139483195
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   ges_class          19716 non-null  object
 1   chauffage_energie  12571 non-null  object
 2   chauffage_systeme  7627 non-null   object
 3   chauffage_mode     13610 non-null  object
dtypes: object(4)
memory usage: 1.1+ MB


In [9]:
df_ventes_filtered[valeurs_manq_resid_quali].describe()

,ges_class,chauffage_energie,chauffage_systeme,chauffage_mode
count,19716,12571,7627,13610
unique,9,10,13,6
top,D,Gaz,Radiateur,Individuel
freq,3943,7787,6254,11186


In [10]:
# Echantillon temoin:
index_temoin = df_ventes_filtered[df_ventes_filtered["chauffage_energie"].isna()].index[0]
df_ventes_filtered.loc[index_temoin]

type_annonceur                    pr
typedebien                         m
typedetransaction                  v
etage                              0
surface                          187
surface_terrain               3182.0
nb_pieces                          9
prix_bien                     282900
mensualiteFinance                  0
balcon                             0
eau                                0
bain                               0
dpeL                               E
dpeC                           251.0
mapCoordonneesLatitude      48.16989
mapCoordonneesLongitude      7.13037
annonce_exclusive                Oui
nb_etages                          2
places_parking                     2
cave                               1
exposition                         0
ges_class                          A
annee_construction              1880
nb_toilettes                       2
ascenseur                          1
nb_logements_copro                 2
charges_copro                  500.0
c

## Etape 3: Cellule récapitulative regroupant toutes les étapes: Remplissage des colonnes quali du X_train et X_test

In [5]:
# Si on veut faire directement l'étape 3, partir du fichier sauvé à l'étape 2 et redéfinir un train / test en enlevant les commentaire ci-dessous 
# import pandas as pd
# df_ventes_filtered = pd.read_csv("df_filled_quanti.csv", index_col = "idannonce")
# X_train = df_ventes_filtered
# X_test = pd.DataFrame(columns = X_train.columns)


def fit_and_or_transform(df_train, df_test = None):
    if df_test is None:
        input_df = df_train
    else:
        input_df = df_test
    valeurs_manq_resid_quali = [col for  col in input_df.select_dtypes(include='object').columns if input_df[col].isna().sum() > 0]
    
    # On va itérer sur chaque ligne avec une valeur manquante
    index_na = list(input_df[input_df[valeurs_manq_resid_quali].isna().any(axis = 1)].index)
    len_index_na = len(index_na)
    print(f"Going to process {len_index_na} records")
    i = 1
    # Valeurs par défaut
    default_mode = {col_quali: df_train[col_quali].mode()[0] for col_quali in valeurs_manq_resid_quali}
    for index in index_na:
        if i % 100 == 0:
            print(f"\rprocessing index {i}/{len_index_na}", end = "")
        i += 1
        latitude = input_df.loc[index, 'mapCoordonneesLatitude']
        longitude = input_df.loc[index, 'mapCoordonneesLongitude']
        nb_piece = input_df.loc[index, 'nb_pieces']
        type_bien = input_df.loc[index, 'typedebien']
    
        # On selectionne les enregistrements du dataframe de train même type avec le même nombre de pièces
        neighbours = df_train[(df_train['nb_pieces'] == nb_piece) & (df_train['typedebien'] == type_bien)]
        # On calcule une fois le vecteur de distance pour l'ensemble des voisins (colonnes quanti à na ou non)
        neighbours_distance = (latitude - neighbours['mapCoordonneesLatitude'])**2 +(longitude - neighbours['mapCoordonneesLongitude'])**2  
        # On itère sur les variables quali manquantes de l'enregistrement
        var_col_quali = input_df.loc[index][valeurs_manq_resid_quali].isna()
        for col_quali in var_col_quali[var_col_quali].index:
            ## On prend les 10 plus proches voisins n'ayant pas la variable à Na
            neighbours_indexes = neighbours_distance[neighbours[col_quali].notna()].sort_values().iloc[:10].index
            if len(neighbours_indexes) > 0:
                input_df.loc[index, col_quali] = neighbours.loc[neighbours_indexes][col_quali].mode()[0]
            else:
                print(f"\nInfo: no nearest neighbours found for index {index} and col {col_quali}")
                input_df.loc[index, col_quali] = default_mode[col_quali]
    print("\n")

def fit_transform(df_train):
    fit_and_or_transform(df_train)

def transform(df_train, df_test):
    fit_and_or_transform(df_train, df_test)

print("Processing X_train")
fit_transform(X_train)
print("Processing X_test")
transform(X_train, X_test)

Processing X_train
Going to process 22474 records
processing index 4300/22474
Info: no nearest neighbours found for index immo-facile-3797119 and col chauffage_systeme
processing index 11200/22474
Info: no nearest neighbours found for index ag681404-326620851 and col ges_class

Info: no nearest neighbours found for index ag681404-326620851 and col chauffage_energie

Info: no nearest neighbours found for index ag681404-326620851 and col chauffage_systeme

Info: no nearest neighbours found for index ag681404-326620851 and col chauffage_mode
processing index 13600/22474
Info: no nearest neighbours found for index hektor-ghisimmobilier-1876 and col chauffage_energie

Info: no nearest neighbours found for index hektor-ghisimmobilier-1876 and col chauffage_mode
processing index 14000/22474
Info: no nearest neighbours found for index guy-hoquet-immo-facile-5532808 and col chauffage_systeme
processing index 20100/22474
Info: no nearest neighbours found for index 157949837 and col chauffage_ene

In [9]:
df_ventes_filtered[valeurs_manq_resid_quali].info()

NameError: name 'valeurs_manq_resid_quali' is not defined

#### Il n'y a plus de valeurs quali manquantes

In [32]:
df_ventes_filtered.loc[index_temoin]

type_annonceur                     pr
typedebien                          m
typedetransaction                   v
etage                               0
surface                           187
surface_terrain                3182.0
nb_pieces                           9
prix_bien                      282900
mensualiteFinance                   0
balcon                              0
eau                                 0
bain                                0
dpeL                                E
dpeC                            251.0
mapCoordonneesLatitude       48.16989
mapCoordonneesLongitude       7.13037
annonce_exclusive                 Oui
nb_etages                           2
places_parking                      2
cave                                1
exposition                          0
ges_class                           A
annee_construction               1880
nb_toilettes                        2
ascenseur                           1
nb_logements_copro                  2
charges_copr

In [33]:
### On sauvegarde le df complété
df_ventes_filtered.to_csv("df_filled_quanti_quali.csv")

#### test de verification, on prend un echantillon plu spetit et on met une valeur test pour le df_train pour vérifier que le remplissage du test se fait bien avec les valeurs du train

In [50]:
import pandas as pd
df_ventes_filtered = pd.read_csv("df_filled_quanti.csv", index_col = "idannonce")

In [51]:
from sklearn.model_selection import train_test_split

In [52]:
df_test = df_ventes_filtered[:1000]

In [53]:
target_feature = 'prix_bien'
target = df_test[target_feature]
data = df_test.drop(target_feature, axis=1)

In [54]:
X_train, X_test, y_train, y_test = train_test_split(data, target, test_size=0.2, random_state=66) 

In [55]:
X_train['chauffage_energie'] = 'Test'

In [56]:
index_test = X_test[X_test['chauffage_energie'].isna()].index

In [57]:
print(f"{X_test.shape[0]}, {len(index_test)}") 

200, 127


In [58]:
print("Processing X_train")
fit_transform(X_train)
print("Processing X_test")
transform(X_train, X_test)

Processing X_train
Going to process 670 records
Info: no nearest neighbours found for index immo-facile-52477509 and col chauffage_systeme
Info: no nearest neighbours found for index ag681088-382101874 and col chauffage_systeme
processing index 100/670Info: no nearest neighbours found for index rodacom-5249976 and col chauffage_systeme
Info: no nearest neighbours found for index rodacom-5249976 and col chauffage_mode
Info: no nearest neighbours found for index ag681158-381115093 and col chauffage_mode
Info: no nearest neighbours found for index ag671791-380740432 and col chauffage_systeme
Info: no nearest neighbours found for index ag671791-380740432 and col chauffage_mode
processing index 200/670Info: no nearest neighbours found for index adapt-immo-75008125577 and col chauffage_systeme
Info: no nearest neighbours found for index adapt-immo-75008125577 and col chauffage_mode
processing index 500/670Info: no nearest neighbours found for index citya-immobilier-6790-TDEM100041 and col ch

In [60]:
X_test.loc[index_test]['chauffage_energie'].value_counts()

chauffage_energie
Test    127
Name: count, dtype: int64